In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import time
from utils import measure_runtime, extract_columns_from_json

# Load environment variables
load_dotenv()

# Initialize GPT-5 via LangChain
llm = ChatOpenAI(
    model="gpt-4o",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Build messages
messages = [
    SystemMessage(content="You are a helpful AI assistant."),
    HumanMessage(content="Explain RAG in one paragraph.")
]

# Invoke model
start = time.perf_counter()
response = llm.invoke(messages)
end = time.perf_counter()

print(response.content)
print(f"Runtime: {end - start:.4f} seconds")


RAG, short for Retrieval-Augmented Generation, is an approach that combines retrieval-based methods with generative models to enhance the quality and relevance of generated outputs. It works by first retrieving relevant documents or passages from a large dataset or knowledge base in response to a query. These retrieved pieces of information are then used as context for a generative model, typically a neural language model, which produces the final output. This integration allows the system to leverage vast amounts of external information, providing more accurate and contextually rich responses, making RAG particularly useful in tasks requiring up-to-date or domain-specific knowledge, such as question answering and conversational AI.
Runtime: 3.8801 seconds


In [2]:
df = pd.read_csv("data/adserver.csv")

In [3]:
df["status"].unique()

array(['FINISHED', 'PENDING'], dtype=object)

In [4]:
#Preprocess
df = df[df["status"] == "FINISHED"]
df = df[["website", "detect_result", "risk_score", "violation_details"]]

In [5]:
df["detect_result"].value_counts()

detect_result
NON_VIOLATION    4176
VIOLATION        1952
REVIEW            264
Name: count, dtype: int64

In [6]:
df.head()

,website,detect_result,risk_score,violation_details
0,kilo.health,NON_VIOLATION,0.28,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
1,sugoimart.com,NON_VIOLATION,0.18,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
2,comfilife.com,NON_VIOLATION,0.22,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
3,omniluxled.com,NON_VIOLATION,0.38,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."
4,310nutrition.com,NON_VIOLATION,0.22,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."


In [7]:
# df[df["detect_result"] == "VIOLATION"].iloc[0]["violation_details"]
# df[df["detect_result"] == "REVIEW"].iloc[0]["violation_details"]

In [8]:
df[(df["detect_result"] == "VIOLATION") & (df["risk_score"] < 0.5)]


,website,detect_result,risk_score,violation_details
5717,playgames365.net,VIOLATION,0.41,"{""violationCategory"":""NON_VIOLATION"",""riskScor..."


In [9]:
target_cols = ["policyViolations", "summary", "detailedBreakdown"]

df = extract_columns_from_json(
    df,
    source_col="violation_details",
    target_cols=target_cols
)

In [10]:
df["policyViolations"].value_counts()

policyViolations
[]                                        6263
[Weaponry]                                  28
[Sex Toys]                                  23
[Political Content]                         19
[Financial Products]                        17
[Low-Quality Affliate or Review Sites]      17
[Pornography]                                8
[Data Collection]                            7
[Dead Links]                                 3
[Graphic Content]                            2
[Sweepstakes]                                2
[Kratom]                                     1
Name: count, dtype: int64

In [11]:
def first_or_nan(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return ""

df["policyViolation"] = df["policyViolations"].apply(first_or_nan)


In [12]:
df[df["policyViolation"] != ""]["risk_score"].value_counts()

risk_score
1.00    125
0.76      1
0.82      1
Name: count, dtype: int64

In [13]:
import re
def count_steps(text: str) -> int:
    if not isinstance(text, str):
        return 0

    # 1) Convert escaped newlines/tabs to real whitespace
    text = re.sub(r"\\[nrt]", " ", text)

    # 2) Normalize all whitespace (Unicode-safe)
    text = re.sub(r"\s+", " ", text)

    # 3) Count "Step <number>" patterns
    return len(re.findall(r"\bStep\s+\d+\b", text, flags=re.IGNORECASE))

df["num_steps"] = df["detailedBreakdown"].apply(count_steps)


In [14]:
#Normally, there are 4 steps:

#Step 1 - On-site review
#Step 2 - Web search and external signals
#Step 3 - Risk scoring per required criteria
#Step 4 - Categorization

In [16]:
llm = ChatOpenAI(
    model="gpt-4o",
    model_kwargs={
        "response_format": {"type": "json_object"}
    }
)

In [35]:
df = pd.read_csv("final_output.csv")
df = df[["website", "detect_result", "risk_score", "summary", "detailedBreakdown", "risk_features"]]

In [36]:
df["detailedBreakdown"]

0       Step 1 – What’s on the URL/landing page (kilo....
1       Step 1 – On‑site content and landing page chec...
2       Step 1 – On‑site review (target URL and landin...
3       Step 1 – On‑site content and landing page\n• T...
4       Step 1 – On-site review (target URL and landin...
                              ...                        
6387    Step 1 – On‑site review of URL content\n• Site...
6388    Step 1 – On‑site review (no policy violations ...
6389    Step 1 – Check actual URL content\n• Result: D...
6390    As-of review date January 9, 2026.\n\nStep 1 –...
6391    Step 1 – On‑site review of coolors.co\n- What ...
Name: detailedBreakdown, Length: 6392, dtype: object

In [32]:
import ast
def extract_amazon_fields(raw_value):
    """
    Extract amazon_present, amazon_score_impact, amazon_evidence
    from a stringified Python dict.
    """
    if not isinstance(raw_value, str):
        return pd.Series([None, None, None])

    try:
        parsed = ast.literal_eval(raw_value)
        amazon = parsed.get("amazon_presence", {})
        return pd.Series([
            amazon.get("present"),
            amazon.get("score_impact"),
            amazon.get("evidence")
        ])
    except (ValueError, SyntaxError):
        return pd.Series([None, None, None])

In [37]:
df[[
    "amazon_present",
    "amazon_score_impact",
    "amazon_evidence"
]] = df["risk_features"].apply(extract_amazon_fields)

In [42]:
df["amazon_score_impact"].value_counts()

amazon_score_impact
 0.10    1402
 0.05    1270
 0.00    1133
 0.15     534
-0.15     358
-0.20     281
-0.10     265
 0.08     190
 0.03     160
-0.05     111
 0.07      89
 0.02      89
 0.12      85
-0.25      73
 0.06      65
-0.12      47
-0.08      41
 0.20      39
 0.04      35
-0.03      22
-0.07      18
-0.18      13
-0.02      12
-1.00       8
-0.30       6
-0.06       5
 0.01       4
 0.25       4
 0.30       3
-0.04       3
 0.50       3
 0.09       3
 0.40       3
 0.60       3
 0.45       2
 1.00       2
-0.14       2
 0.70       1
-0.35       1
 5.00       1
 0.55       1
-0.22       1
-0.16       1
-0.17       1
 0.18       1
 0.22       1
Name: count, dtype: int64